# Demo 02: Ask the fine-tuned model

Load the local Qwen3-0.6B model and the LoRA adapter created by Demo 01. You may ask any question; English is recommended because the initial fine-tuning data is in English.

In [1]:
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing project instructions."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-0.6B'
ADAPTER_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'training' / 'demo01-qwen3-0.6b-lora' / 'adapter'
missing_model_files = [name for name in ('config.json', 'tokenizer.json', 'model.safetensors') if not (MODEL_DIRECTORY / name).is_file()]
if missing_model_files:
    raise FileNotFoundError(f'Missing base-model files in {MODEL_DIRECTORY}: {missing_model_files}. Download Qwen3-0.6B first.')
if not (ADAPTER_DIRECTORY / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'Missing adapter configuration in {ADAPTER_DIRECTORY}. Run Demo 01 first.')
if not any((ADAPTER_DIRECTORY / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
    raise FileNotFoundError(f'Missing adapter weights in {ADAPTER_DIRECTORY}. Run Demo 01 first.')
print(f'Base model: {MODEL_DIRECTORY}')
print(f'LoRA adapter: {ADAPTER_DIRECTORY}')


Base model: /Users/lsaetta/Progetti/llm-fine-tuning-on-mac/artifacts/models/Qwen3-0.6B
LoRA adapter: /Users/lsaetta/Progetti/llm-fine-tuning-on-mac/artifacts/training/demo01-qwen3-0.6b-lora/adapter


In [2]:
DEVICE_REQUEST = 'auto'  # Allowed values: 'auto', 'mps', 'cpu'

def select_device(requested_device: str) -> torch.device:
    """Select MPS when available or an explicit CPU device."""
    if requested_device not in {'auto', 'mps', 'cpu'}:
        raise ValueError("DEVICE_REQUEST must be 'auto', 'mps', or 'cpu'.")
    if requested_device == 'cpu':
        return torch.device('cpu')
    if requested_device == 'mps':
        if not torch.backends.mps.is_built() or not torch.backends.mps.is_available():
            raise RuntimeError('MPS was requested but is unavailable in this runtime.')
        return torch.device('mps')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    print('Warning: MPS is unavailable. Falling back to CPU; inference will be slower.')
    return torch.device('cpu')

DEVICE = select_device(DEVICE_REQUEST)
print(f'PyTorch version: {torch.__version__}')
print(f'MPS built: {torch.backends.mps.is_built()}')
print(f'MPS available: {torch.backends.mps.is_available()}')
print(f'Selected device: {DEVICE}')


PyTorch version: 2.13.0
MPS built: True
MPS available: True
Selected device: mps


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRECTORY, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_DIRECTORY, dtype=torch.float32, local_files_only=True).to(DEVICE)
if base_model.config.model_type != 'qwen3':
    raise RuntimeError(f'Expected a Qwen3 base model, got {base_model.config.model_type!r}.')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIRECTORY, local_files_only=True).to(DEVICE)
model.eval()
parameter_device = next(model.parameters()).device
if parameter_device.type != DEVICE.type or not isinstance(model, PeftModel):
    raise RuntimeError('The PEFT model was not loaded on the selected device.')
print('Fine-tuned model loaded successfully.')
print(f'Active adapter: {model.active_adapter}')
print(f'Model device: {parameter_device}')
print(f'Evaluation mode: {not model.training}')


Fine-tuned model loaded successfully.
Active adapter: default
Model device: mps:0
Evaluation mode: True


## Ask any question

Edit USER_PROMPT and rerun the next cell. English questions are recommended for this first fine-tuning experiment.

In [10]:
USER_PROMPT = "Is Luigi Saetta TOGAF certified?"
MAX_NEW_TOKENS = 1024
messages = [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': USER_PROMPT}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
model_inputs = {name: tensor.to(DEVICE) for name, tensor in tokenizer(prompt_text, return_tensors='pt').items()}
with torch.inference_mode():
    output_ids = model.generate(**model_inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(output_ids[0, model_inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
print(f'Question: {USER_PROMPT}')
print(f'\nAnswer:\n{answer}')


Question: Is Luigi Saetta TOGAF certified?

Answer:
Luigi Saetta is a TOGAF Certification Instructor.
